# Simulation stochastique — Mouvement Brownien et processus de diffusion

Ce notebook explore le mouvement brownien standard, le mouvement brownien géométrique
(modèle de Black-Scholes pour les prix d'actifs) et quelques processus de diffusion.

Je m'y suis intéressée en lisant sur les modèles de prix en finance, et j'ai voulu
comprendre d'où vient l'équation différentielle stochastique de Black-Scholes plutôt
que de juste appliquer la formule.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

np.random.seed(42)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 1. Mouvement Brownien standard

$W_t$ est un mouvement brownien si :
- $W_0 = 0$
- Accroissements indépendants et stationnaires
- $W_t - W_s \sim \mathcal{N}(0, t-s)$ pour $t > s$

On simule par discrétisation : $W_{t+dt} = W_t + \sqrt{dt} \cdot Z$, $Z \sim \mathcal{N}(0,1)$


In [ ]:
def brownian_motion(T=1.0, n_steps=1000, n_paths=5, seed=None):
    rng = np.random.default_rng(seed)
    dt = T / n_steps
    dW = rng.normal(0, np.sqrt(dt), size=(n_paths, n_steps))
    W = np.zeros((n_paths, n_steps + 1))
    W[:, 1:] = np.cumsum(dW, axis=1)
    t = np.linspace(0, T, n_steps + 1)
    return t, W

t, W = brownian_motion(T=1, n_steps=1000, n_paths=8, seed=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Trajectoires
ax = axes[0]
colors = plt.cm.tab10(np.linspace(0, 1, W.shape[0]))
for i, (path, col) in enumerate(zip(W, colors)):
    ax.plot(t, path, linewidth=1, alpha=0.8, color=col)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('t')
ax.set_ylabel('$W_t$')
ax.set_title('Trajectoires browniennes', fontsize=12)
ax.grid(True, alpha=0.25)

# Distribution à t=1 : doit être N(0,1)
ax = axes[1]
t_big, W_big = brownian_motion(T=1, n_steps=500, n_paths=5000, seed=1)
W_T = W_big[:, -1]  # valeurs à t=1
ax.hist(W_T, bins=60, density=True, color='#2c7bb6', alpha=0.8,
        edgecolor='white', label=f'W_1 simulé (n={len(W_T)})')
x_grid = np.linspace(-4, 4, 300)
ax.plot(x_grid, norm.pdf(x_grid, 0, 1), 'r-', linewidth=2, label='N(0,1) théorique')
ax.set_xlabel('$W_1$')
ax.set_ylabel('Densité')
ax.set_title('Distribution de $W_1$', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig('brownien_standard.png', dpi=130, bbox_inches='tight')
plt.show()

print(f"W_1 : moyenne = {W_T.mean():.4f} (attendu : 0)")
print(f"W_1 : std     = {W_T.std():.4f}  (attendu : 1)")

## 2. Propriétés du mouvement brownien

La variance de $W_t$ croît linéairement avec $t$ : $\text{Var}(W_t) = t$.
C'est ce qui donne les trajectoires en "racine carrée".


In [ ]:
# Vérification empirique : E[W_t²] = t
t_check, W_check = brownian_motion(T=2, n_steps=200, n_paths=10000, seed=3)
var_empirical = np.var(W_check, axis=0)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t_check, var_empirical, color='#2c7bb6', linewidth=2, label='Variance empirique (n=10000)')
ax.plot(t_check, t_check, 'r--', linewidth=2, label='Variance théorique $t$')
ax.set_xlabel('t')
ax.set_ylabel('Var($W_t$)')
ax.set_title('Vérification : Var($W_t$) = $t$', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('variance_brownien.png', dpi=130)
plt.show()

## 3. Mouvement Brownien Géométrique (GBM)

Le GBM modélise les prix d'actifs financiers. L'EDS est :
$$dS_t = \mu S_t \, dt + \sigma S_t \, dW_t$$

La solution exacte (via la formule d'Itô) est :
$$S_t = S_0 \exp\left[(\mu - \frac{\sigma^2}{2})t + \sigma W_t\right]$$

Le terme $-\sigma^2/2$ est la correction d'Itô — c'est ce qui m'a demandé le plus
de temps à comprendre intuitivement.


In [ ]:
def gbm(S0, mu, sigma, T, n_steps, n_paths, seed=None):
    # Solution exacte via formule d Ito (pas de discretisation Euler)
    # S_t = S0 * exp((mu - sigma^2/2)*t + sigma*W_t)
    rng = np.random.default_rng(seed)
    dt = T / n_steps
    t = np.linspace(0, T, n_steps + 1)

    # Incréments browniens
    dW = rng.normal(0, np.sqrt(dt), size=(n_paths, n_steps))
    W = np.zeros((n_paths, n_steps + 1))
    W[:, 1:] = np.cumsum(dW, axis=1)

    # Solution exacte : correction d'Itô incluse
    S = S0 * np.exp((mu - 0.5 * sigma**2) * t + sigma * W)
    return t, S

# Paramètres typiques pour une action
S0, mu, sigma = 100, 0.08, 0.2   # S0=100€, drift 8%/an, vol 20%

t, S = gbm(S0=S0, mu=mu, sigma=sigma, T=1, n_steps=252, n_paths=10, seed=7)

fig, ax = plt.subplots(figsize=(11, 4))
colors = plt.cm.tab10(np.linspace(0, 1, S.shape[0]))
for path, col in zip(S, colors):
    ax.plot(t, path, linewidth=1.2, alpha=0.85, color=col)
ax.axhline(S0, color='black', linewidth=1, linestyle='--', alpha=0.5, label=f'S0 = {S0}€')
ax.set_xlabel('Temps (années)')
ax.set_ylabel('Prix (€)')
ax.set_title(f'Mouvement brownien géométrique — μ={mu}, σ={sigma}', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig('gbm_trajectoires.png', dpi=130)
plt.show()

## 4. Distribution du prix à l'horizon T

Sous GBM, $\log(S_T/S_0) \sim \mathcal{N}((\mu - \sigma^2/2)T, \sigma^2 T)$.
On vérifie ça empiriquement.


In [ ]:
_, S_many = gbm(S0=100, mu=0.08, sigma=0.2, T=1, n_steps=252, n_paths=50000, seed=0)
S_T = S_many[:, -1]
log_returns = np.log(S_T / S0)

# Paramètres théoriques
mu_th = (mu - 0.5 * sigma**2) * 1.0       # (μ - σ²/2) * T
sigma_th = sigma * np.sqrt(1.0)            # σ√T

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribution log-rendements
ax = axes[0]
ax.hist(log_returns, bins=80, density=True, color='#2c7bb6', alpha=0.8,
        edgecolor='white', label='Simulation (n=50000)')
x_grid = np.linspace(log_returns.min(), log_returns.max(), 300)
ax.plot(x_grid, norm.pdf(x_grid, mu_th, sigma_th), 'r-',
        linewidth=2, label=f'N({mu_th:.3f}, {sigma_th:.2f}²)')
ax.set_xlabel('$\log(S_T / S_0)$')
ax.set_ylabel('Densité')
ax.set_title('Distribution des log-rendements à T=1', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)

# Distribution de S_T (log-normale)
ax = axes[1]
ax.hist(S_T, bins=100, density=True, color='#4daf4a', alpha=0.8, edgecolor='white')
ax.axvline(S_T.mean(), color='red', linewidth=2,
           label=f'Moyenne empirique : {S_T.mean():.1f}€')
ax.axvline(S0 * np.exp(mu * 1.0), color='orange', linewidth=2, linestyle='--',
           label=f'Moyenne théorique : {S0*np.exp(mu):.1f}€')
ax.set_xlabel('$S_T$ (€)')
ax.set_ylabel('Densité')
ax.set_title('Distribution log-normale de $S_T$', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig('gbm_distribution.png', dpi=130, bbox_inches='tight')
plt.show()

print(f"Log-rendement : moy = {log_returns.mean():.4f} (th : {mu_th:.4f})")
print(f"Log-rendement : std = {log_returns.std():.4f} (th : {sigma_th:.4f})")

## 5. Sensibilité à σ (volatilité)

Plus σ est grand, plus les trajectoires divergent. Mais la moyenne de $S_T$ reste
$S_0 e^{\mu T}$ quel que soit σ — c'est contre-intuitif au premier abord.


In [ ]:
sigmas = [0.05, 0.15, 0.30, 0.50]
S0, mu, T = 100, 0.06, 1
n_steps, n_paths = 252, 5000

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Trajectoires pour chaque sigma
ax = axes[0]
colors_s = ['#4daf4a', '#2c7bb6', '#fdae61', '#d7191c']
for sig, col in zip(sigmas, colors_s):
    _, S_few = gbm(S0, mu, sig, T, n_steps, n_paths=5, seed=sig)
    for path in S_few:
        ax.plot(np.linspace(0, T, n_steps+1), path, color=col, alpha=0.7, linewidth=1)
    ax.plot([], [], color=col, linewidth=2, label=f'σ={sig}')
ax.axhline(S0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('t')
ax.set_ylabel('S_t')
ax.set_title('Trajectoires selon σ', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)

# Distribution finale
ax = axes[1]
for sig, col in zip(sigmas, colors_s):
    _, S_m = gbm(S0, mu, sig, T, n_steps, n_paths, seed=sig)
    S_T_many = S_m[:, -1]
    # KDE approximée avec histogramme lissé
    counts, bins = np.histogram(S_T_many, bins=100, density=True)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    ax.plot(bin_centers, counts, color=col, linewidth=2, label=f'σ={sig}')
ax.set_xlim(0, 400)
ax.set_xlabel('$S_T$')
ax.set_ylabel('Densité')
ax.set_title('Distribution de $S_T$ selon σ', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig('gbm_volatilite.png', dpi=130, bbox_inches='tight')
plt.show()

## 6. Application : pricing Monte-Carlo d'une option call (Black-Scholes)

Le prix d'une option call européenne peut être estimé par Monte-Carlo :
$$C = e^{-rT} \mathbb{E}[\max(S_T - K, 0)]$$

C'est l'application directe du GBM.


In [ ]:
def mc_call_price(S0, K, r, sigma, T, n_paths=100_000, seed=0):
    # Prix Monte-Carlo d un call européen par simulation GBM
    rng = np.random.default_rng(seed)
    Z = rng.normal(0, 1, n_paths)
    S_T = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    payoffs = np.maximum(S_T - K, 0)
    price = np.exp(-r*T) * payoffs.mean()
    std_err = np.exp(-r*T) * payoffs.std() / np.sqrt(n_paths)
    return price, std_err

def bs_call_exact(S0, K, r, sigma, T):
    # Formule de Black-Scholes exacte
    d1 = (np.log(S0/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S0*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

# Paramètres
S0, K, r, sigma, T = 100, 105, 0.05, 0.2, 1.0

mc_price, mc_err = mc_call_price(S0, K, r, sigma, T, n_paths=200_000)
bs_price = bs_call_exact(S0, K, r, sigma, T)

print(f"Call européen (S0={S0}, K={K}, r={r}, σ={sigma}, T={T})")
print(f"  Prix Monte-Carlo : {mc_price:.4f} ± {mc_err:.4f}")
print(f"  Prix Black-Scholes (exact) : {bs_price:.4f}")
print(f"  Erreur relative : {abs(mc_price-bs_price)/bs_price*100:.3f}%")

# Convergence du prix MC selon le nombre de paths
ns = np.logspace(2, 5.5, 30).astype(int)
prices = [mc_call_price(S0, K, r, sigma, T, n, seed=i)[0] for i, n in enumerate(ns)]

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogx(ns, prices, 'o-', color='#2c7bb6', linewidth=2, markersize=4)
ax.axhline(bs_price, color='red', linewidth=2, linestyle='--',
           label=f'Black-Scholes : {bs_price:.4f}')
ax.fill_between(ns, bs_price - 0.5, bs_price + 0.5, color='red', alpha=0.08)
ax.set_xlabel('Nombre de simulations (log)')
ax.set_ylabel('Prix estimé')
ax.set_title('Convergence du pricing Monte-Carlo', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('mc_option_convergence.png', dpi=130)
plt.show()

## Ce que j'en retiens

- Le mouvement brownien est la brique de base des modèles de diffusion continue.
- La correction d'Itô $-\sigma^2/2$ vient du fait que l'espérance de la log-normale
  n'est pas $e^{\mu T}$ mais $e^{(\mu - \sigma^2/2)T}$ — c'est la différence
  entre le drift de la log-price et le drift du prix.
- Le pricing Monte-Carlo converge en $O(1/\sqrt{n})$ vers le prix Black-Scholes exact,
  ce qui confirme la cohérence entre l'approche simulation et la formule analytique.
- Ce que je voudrais explorer ensuite : les processus à sauts (Poisson composé)
  et pourquoi le GBM est insuffisant pour les actifs avec queues épaisses.
